# GEE Pre-processing: Friction Surface Data Acquisition

This notebook pulls geospatial data from Google Earth Engine (GEE) and external
sources, processes it, and exports aligned rasters for the Alaska fuel delivery
friction surface system.

**All outputs are in EPSG:3413** (WGS 84 / NSIDC Sea Ice Polar Stereographic North),
clipped to Alaska, at **150m resolution** with consistent pixel alignment.

### Output Rasters
| File | Source | Description |
|------|--------|-------------|
| `lulc_alaska_modal.tif` | Dynamic World | Modal land cover classification (classes 0-8) |
| `slope_alaska.tif` | FabDEM | Slope in degrees |
| `dem_alaska.tif` | FabDEM | Raw elevation values |
| `permafrost_alaska.tif` | Obu et al. 2019 | Permafrost zones (0-3) |
| `roads_presence_alaska.tif` | GRIP4 | Binary road presence |
| `roads_type_alaska.tif` | AK DOT + USGS NTD | Road surface type (paved/gravel/dirt) |
| `rivers_alaska.tif` | NHD | Major/minor river classification |

### Output Vectors
| File | Source | Description |
|------|--------|-------------|
| `airports_alaska.geojson` | FAA | Airport point locations |
| `ports_alaska.geojson` | AK DOT&PF | Port locations with type |
| `facilities_alaska.geojson` | Bulk Fuel CSV | Fuel facility sites |

**References:**
- Trochim et al. (review) — friction surface methodology
- Atkinson et al., 2005 — slope classification
- Obu et al., 2019 — permafrost zonation

## 1. Setup & Authentication

In [ ]:
# --- Setup ---
# On Colab: pip install earthengine-api geopandas rasterio pyproj
# !pip install earthengine-api geopandas rasterio pyproj

import ee
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, mapping

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize(project='your-project-id')  # <-- Replace with your GEE project ID

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
TARGET_CRS = 'EPSG:3413'   # NSIDC Sea Ice Polar Stereographic North
TARGET_SCALE = 150          # metres
DRIVE_FOLDER = 'friction_surface_exports'  # Google Drive folder for exports

# Alaska bounding box (generous, in WGS84)
ALASKA_BBOX = ee.Geometry.Rectangle([-180, 51, -129, 72])

# Alaska state boundary from TIGER/Census for precise clipping
alaska_states = ee.FeatureCollection('TIGER/2018/States')
alaska_boundary = alaska_states.filter(ee.Filter.eq('NAME', 'Alaska')).geometry()

# Shared CRS transform for pixel alignment across all exports.
# This ensures all rasters snap to the same grid.
# Format: [xScale, xShearing, xTranslation, yShearing, yScale, yTranslation]
# We use a fixed origin so all exports align perfectly.
CRS_TRANSFORM = [150, 0, -3000000, 0, -150, 3000000]

print(f"Target CRS: {TARGET_CRS}")
print(f"Target scale: {TARGET_SCALE}m")
print(f"Drive folder: {DRIVE_FOLDER}")
print("GEE initialized successfully.")

In [ ]:
def export_aligned_raster(image, description, filename, region=None,
                          scale=TARGET_SCALE, crs=TARGET_CRS,
                          crs_transform=CRS_TRANSFORM, folder=DRIVE_FOLDER,
                          max_pixels=1e10):
    """Export a GEE image to Google Drive with consistent alignment.

    All exports use the same CRS, scale, and crsTransform so output
    rasters are perfectly pixel-aligned.

    Args:
        image: ee.Image to export
        description: Task description (shown in GEE Tasks tab)
        filename: Output filename (without extension)
        region: ee.Geometry for clipping (default: Alaska boundary)
        scale: Pixel size in metres
        crs: Coordinate reference system
        crs_transform: Affine transform for pixel alignment
        folder: Google Drive folder name
        max_pixels: Maximum number of pixels to export

    Returns:
        ee.batch.Task: The started export task
    """
    if region is None:
        region = alaska_boundary

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=filename,
        region=region,
        crs=crs,
        crsTransform=crs_transform,
        maxPixels=int(max_pixels),
        fileFormat='GeoTIFF',
    )
    task.start()
    print(f"Export started: {description} -> {folder}/{filename}.tif")
    return task


# Collect all tasks for monitoring
export_tasks = []

## 2. LULC — Dynamic World (Modal Land Cover)

In [ ]:
# Dynamic World v1 — compute per-pixel modal (most common) class over 2023
# Classes: 0=water, 1=trees, 2=grass, 3=flooded_vegetation, 4=crops,
#          5=shrub_and_scrub, 6=built, 7=bare, 8=snow_and_ice

dw_collection = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterDate('2023-01-01', '2023-12-31')
    .filterBounds(alaska_boundary)
    .select('label')
)

# Compute per-pixel mode (most frequent class across the year)
lulc_modal = dw_collection.mode().clip(alaska_boundary).rename('lulc')

# Export
task = export_aligned_raster(
    image=lulc_modal.toInt8(),
    description='LULC_Alaska_Modal_2023',
    filename='lulc_alaska_modal',
)
export_tasks.append(task)

print("Dynamic World LULC classes:")
print("  0=water, 1=trees, 2=grass, 3=flooded_vegetation, 4=crops")
print("  5=shrub_and_scrub, 6=built, 7=bare, 8=snow_and_ice")

## 3. DEM & Slope — FabDEM

In [ ]:
# FabDEM — Forest And Buildings removed Copernicus DEM
# Source: https://gee-community-catalog.org/projects/fabdem/
fabdem = ee.ImageCollection('projects/sat-io/open-datasets/FABDEM').mosaic()

dem_alaska = fabdem.clip(alaska_boundary).rename('elevation')

# Export raw DEM (elevation in metres)
task_dem = export_aligned_raster(
    image=dem_alaska.toFloat(),
    description='DEM_Alaska_FabDEM',
    filename='dem_alaska',
)
export_tasks.append(task_dem)

# Compute slope in degrees
slope_alaska = ee.Terrain.slope(dem_alaska).rename('slope')

# Export slope
task_slope = export_aligned_raster(
    image=slope_alaska.toFloat(),
    description='Slope_Alaska_FabDEM',
    filename='slope_alaska',
)
export_tasks.append(task_slope)

print("DEM and slope exports started.")

## 4. Permafrost — Obu et al., 2019

In [ ]:
# Obu et al. 2019 — Global Permafrost Zonation Index Map
# Source: https://gee-community-catalog.org/projects/pzim/
# The PZIM values range from 0 to 1, representing permafrost probability.
# We reclassify into 4 zones:
#   0 = none (PZIM < 0.1)
#   1 = sporadic (0.1 <= PZIM < 0.5)
#   2 = discontinuous (0.5 <= PZIM < 0.9)
#   3 = continuous (PZIM >= 0.9)

pzim = ee.Image('projects/sat-io/open-datasets/landform/global_permafrost_zonation')

# Reclassify
permafrost = (
    pzim
    .where(pzim.lt(0.1), 0)
    .where(pzim.gte(0.1).And(pzim.lt(0.5)), 1)
    .where(pzim.gte(0.5).And(pzim.lt(0.9)), 2)
    .where(pzim.gte(0.9), 3)
    .clip(alaska_boundary)
    .rename('permafrost')
)

# Export
task = export_aligned_raster(
    image=permafrost.toInt8(),
    description='Permafrost_Alaska_Obu2019',
    filename='permafrost_alaska',
)
export_tasks.append(task)

print("Permafrost zones: 0=none, 1=sporadic, 2=discontinuous, 3=continuous")

## 5. Roads — GRIP4 (Presence)

In [ ]:
# GRIP4 — Global Roads Inventory Project (version 4)
# Source: https://gee-community-catalog.org/projects/grip4/
# Rasterize as binary: 1 = road present, 0 = no road

grip4 = ee.Image('projects/sat-io/open-datasets/GRIP4/grip4_total_TP')

# Any GRIP4 value > 0 indicates road presence
roads_presence = (
    grip4
    .gt(0)
    .selfMask()  # mask out 0 values (no road)
    .clip(alaska_boundary)
    .rename('road_presence')
)

# Export (unmask to fill NoData with 0 for non-road cells)
task = export_aligned_raster(
    image=roads_presence.unmask(0).toInt8(),
    description='Roads_Presence_Alaska_GRIP4',
    filename='roads_presence_alaska',
)
export_tasks.append(task)

print("GRIP4 road presence: 1=road, 0=no road")

## 6. Roads — AK DOT Surface Type

AK DOT road data is served from an ArcGIS REST service, not natively in GEE.
We download it as GeoJSON, reclassify surface types, and rasterize locally.

In [ ]:
# AK DOT Roads — surface type classification
# Source: https://gis.data.alaska.gov/datasets/AKDOT::roads-akdot/
#
# The ArcGIS REST API endpoint for this layer:
AKDOT_ROADS_URL = (
    "https://services prior.prior.prior/arcgis/rest/services/prior/FeatureServer/0/query"
)
# NOTE: The exact URL may change. Use the actual REST endpoint from:
# https://gis.data.alaska.gov/datasets/AKDOT::roads-akdot/
#
# Alternatively, download the shapefile manually from the AK DOT data portal
# and place it in the working directory.

import requests
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import rasterio
from pyproj import Transformer

def fetch_akdot_roads(output_path='akdot_roads_alaska.geojson'):
    """Fetch AK DOT roads from ArcGIS REST API.

    If the API is unavailable, provide instructions for manual download.
    """
    # Try fetching from ArcGIS REST API with pagination
    base_url = (
        "https://services.prior.prior/arcgis/rest/services/"
        "prior/FeatureServer/0/query"
    )
    # NOTE: Replace with the actual ArcGIS REST endpoint. You can find it by:
    # 1. Go to https://gis.data.alaska.gov/datasets/AKDOT::roads-akdot/
    # 2. Click "API" or "View Full Details"
    # 3. Copy the REST endpoint URL

    print("=" * 60)
    print("AK DOT Roads — Manual Download Required")
    print("=" * 60)
    print()
    print("1. Go to: https://gis.data.alaska.gov/datasets/AKDOT::roads-akdot/")
    print("2. Click 'Download' and select GeoJSON or Shapefile format")
    print("3. Save as 'akdot_roads_alaska.geojson' in the working directory")
    print()
    print("OR use the ArcGIS REST API query endpoint directly.")
    print()

    if os.path.exists(output_path):
        print(f"Found existing file: {output_path}")
        return gpd.read_file(output_path)
    else:
        print(f"File not found: {output_path}")
        print("Please download the data manually and re-run this cell.")
        return None


def rasterize_road_types(roads_gdf, reference_raster_path, output_path):
    """Rasterize road surface types to match the reference raster grid.

    Surface type mapping:
        Paved / Asphalt / Concrete -> 1
        Gravel                     -> 2
        Dirt / Earth / Unknown     -> 3

    Args:
        roads_gdf: GeoDataFrame with road geometries and surface type column
        reference_raster_path: Path to a reference raster for grid alignment
        output_path: Output raster path
    """
    # Identify the surface type column (common names in AK DOT data)
    surface_col = None
    for col in ['SURFACE', 'SurfaceTyp', 'SURF_TYPE', 'Surface', 'surface_type']:
        if col in roads_gdf.columns:
            surface_col = col
            break

    if surface_col is None:
        print(f"Available columns: {list(roads_gdf.columns)}")
        print("Could not find surface type column. Using column list above to identify it.")
        print("Set surface_col manually and re-run.")
        return

    # Map surface types to friction categories
    paved_keywords = ['paved', 'asphalt', 'concrete', 'bituminous', 'sealed']
    gravel_keywords = ['gravel', 'aggregate', 'crushed']
    # Everything else maps to dirt (3)

    def classify_surface(val):
        if pd.isna(val):
            return 3  # unknown -> dirt
        val_lower = str(val).lower()
        if any(k in val_lower for k in paved_keywords):
            return 1
        elif any(k in val_lower for k in gravel_keywords):
            return 2
        else:
            return 3

    roads_gdf = roads_gdf.copy()
    roads_gdf['friction_class'] = roads_gdf[surface_col].apply(classify_surface)

    # Reproject to EPSG:3413
    roads_3413 = roads_gdf.to_crs('EPSG:3413')

    # Read reference raster for grid alignment
    with rasterio.open(reference_raster_path) as ref:
        out_shape = (ref.height, ref.width)
        out_transform = ref.transform
        out_crs = ref.crs

    # Rasterize
    shapes = [(geom, val) for geom, val in zip(
        roads_3413.geometry, roads_3413['friction_class']
    ) if geom is not None]

    road_type_raster = rasterize(
        shapes,
        out_shape=out_shape,
        transform=out_transform,
        fill=0,  # 0 = no road
        dtype='int8',
    )

    # Write output
    profile = {
        'driver': 'GTiff',
        'dtype': 'int8',
        'width': out_shape[1],
        'height': out_shape[0],
        'count': 1,
        'crs': out_crs,
        'transform': out_transform,
        'compress': 'lzw',
    }
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(road_type_raster, 1)

    print(f"Road type raster saved: {output_path}")
    print(f"  Shape: {out_shape}, CRS: {out_crs}")
    unique, counts = np.unique(road_type_raster[road_type_raster > 0], return_counts=True)
    for u, c in zip(unique, counts):
        labels = {1: 'paved', 2: 'gravel', 3: 'dirt'}
        print(f"  Class {u} ({labels.get(u, '?')}): {c:,} pixels")


# Attempt to load/download roads
roads_gdf = fetch_akdot_roads()

# NOTE: After roads_presence_alaska.tif is downloaded from GEE,
# run this to create the road type raster:
# rasterize_road_types(roads_gdf, 'roads_presence_alaska.tif', 'roads_type_alaska.tif')

## 7. Rivers — NHD Hydrography

NHD is not natively in GEE. Two approaches are provided:
- **Option A (preferred):** Download NHD Alaska shapefile, upload to GEE as asset, and rasterize
- **Option B:** Use GEE's JRC Global Surface Water as a proxy for major water bodies

In [ ]:
# NHD Rivers — Rasterization
# Download NHD Alaska from:
# https://prd-tnm.s3.amazonaws.com/index.html?prefix=StagedProducts/Hydrography/NHD/State/Shape/
#
# After download, the shapefile can be:
# 1. Uploaded as a GEE asset and rasterized in GEE, OR
# 2. Rasterized locally with rasterio (shown below)

def rasterize_nhd_rivers(nhd_flowlines_path, reference_raster_path, output_path):
    """Rasterize NHD flowlines into major (1) and minor (2) rivers.

    Major rivers: FCode in navigable categories (Strahler order >= 4 or
    named rivers with large stream order)
    Minor rivers: All other flowlines

    Args:
        nhd_flowlines_path: Path to NHDFlowline shapefile
        reference_raster_path: Path to reference raster for grid alignment
        output_path: Output raster path
    """
    print(f"Loading NHD flowlines: {nhd_flowlines_path}")
    flowlines = gpd.read_file(nhd_flowlines_path)

    # Reproject to EPSG:3413
    flowlines_3413 = flowlines.to_crs('EPSG:3413')

    # Classify major vs minor rivers
    # NHD FCode values for navigable waterways:
    #   46006 = Stream/River (perennial)
    #   55800 = Artificial Path (through lakes)
    #   33600 = Canal/Ditch
    # We use StreamOrde (Strahler order) if available
    if 'StreamOrde' in flowlines_3413.columns:
        flowlines_3413['river_class'] = np.where(
            flowlines_3413['StreamOrde'] >= 4, 1, 2  # 1=major, 2=minor
        )
    elif 'GNIS_Name' in flowlines_3413.columns:
        # Major Alaska rivers by name
        major_rivers = [
            'Yukon', 'Kuskokwim', 'Tanana', 'Copper', 'Susitna',
            'Nushagak', 'Kvichak', 'Stikine', 'Colville', 'Noatak',
            'Kobuk', 'Porcupine', 'Innoko', 'Koyukuk',
        ]
        is_major = flowlines_3413['GNIS_Name'].str.contains(
            '|'.join(major_rivers), case=False, na=False
        )
        flowlines_3413['river_class'] = np.where(is_major, 1, 2)
    else:
        # Default: all minor
        flowlines_3413['river_class'] = 2
        print("Warning: No StreamOrde or GNIS_Name column. All rivers classified as minor.")

    # Read reference raster for grid alignment
    with rasterio.open(reference_raster_path) as ref:
        out_shape = (ref.height, ref.width)
        out_transform = ref.transform
        out_crs = ref.crs

    # Rasterize (major rivers first so they overwrite minor where overlapping)
    shapes_minor = [(geom, 2) for geom, cls in zip(
        flowlines_3413.geometry, flowlines_3413['river_class']
    ) if geom is not None and cls == 2]

    shapes_major = [(geom, 1) for geom, cls in zip(
        flowlines_3413.geometry, flowlines_3413['river_class']
    ) if geom is not None and cls == 1]

    # Rasterize minor first, then burn major on top
    river_raster = rasterize(
        shapes_minor + shapes_major,
        out_shape=out_shape,
        transform=out_transform,
        fill=0,  # 0 = no river (will be set to NoData)
        dtype='int8',
    )

    # Write output (0 -> NoData)
    profile = {
        'driver': 'GTiff',
        'dtype': 'int8',
        'width': out_shape[1],
        'height': out_shape[0],
        'count': 1,
        'crs': out_crs,
        'transform': out_transform,
        'compress': 'lzw',
        'nodata': 0,
    }
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(river_raster, 1)

    major_count = np.sum(river_raster == 1)
    minor_count = np.sum(river_raster == 2)
    print(f"River raster saved: {output_path}")
    print(f"  Major navigable (1): {major_count:,} pixels")
    print(f"  Minor (2): {minor_count:,} pixels")


# --- Option B: JRC Global Surface Water as proxy ---
# If NHD is unavailable, use JRC to identify permanent water bodies
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
water_occurrence = jrc.select('occurrence')

# Permanent water (>75% occurrence) as major, seasonal (25-75%) as minor
rivers_proxy = (
    ee.Image(0)
    .where(water_occurrence.gte(75), 1)   # major / permanent
    .where(water_occurrence.gte(25).And(water_occurrence.lt(75)), 2)  # minor
    .selfMask()
    .clip(alaska_boundary)
    .rename('rivers')
)

task = export_aligned_raster(
    image=rivers_proxy.toInt8(),
    description='Rivers_Alaska_JRC_proxy',
    filename='rivers_alaska',
)
export_tasks.append(task)

print("Rivers exported using JRC Global Surface Water as proxy.")
print("For higher quality, download NHD and use rasterize_nhd_rivers().")
print("NHD download: https://prd-tnm.s3.amazonaws.com/index.html?prefix=StagedProducts/Hydrography/NHD/State/Shape/")

## 8. Airports — FAA

In [ ]:
# FAA Airport Locations
# The FAA maintains airport data that can be accessed via:
# 1. GEE: FAA/NASR/2024/airports (if available)
# 2. Direct download from FAA NASR data
# 3. OpenFlights dataset (backup)

# Try GEE asset first
try:
    faa_airports = ee.FeatureCollection('FAA/NASR/2024/airports')
    # Filter to Alaska (state code 'AK')
    ak_airports = faa_airports.filter(ee.Filter.eq('STATE', 'AK'))
    print(f"FAA airports in Alaska: {ak_airports.size().getInfo()}")

    # Export as GeoJSON
    # Convert to pandas via getInfo (small dataset, OK for direct download)
    airport_info = ak_airports.getInfo()
    features = airport_info['features']

    airport_records = []
    for f in features:
        props = f['properties']
        coords = f['geometry']['coordinates']
        airport_records.append({
            'name': props.get('ARPT_NAME', ''),
            'faa_id': props.get('ARPT_ID', ''),
            'city': props.get('CITY', ''),
            'state': props.get('STATE', 'AK'),
            'longitude': coords[0],
            'latitude': coords[1],
        })

    airports_gdf = gpd.GeoDataFrame(
        airport_records,
        geometry=gpd.points_from_xy(
            [r['longitude'] for r in airport_records],
            [r['latitude'] for r in airport_records],
        ),
        crs='EPSG:4326',
    )

except Exception as e:
    print(f"GEE FAA asset not available: {e}")
    print("Using OpenFlights as backup source...")

    # OpenFlights backup — covers major airports
    # Download from https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat
    openflights_cols = [
        'id', 'name', 'city', 'country', 'iata', 'icao',
        'latitude', 'longitude', 'altitude', 'timezone', 'dst',
        'tz_database', 'type', 'source'
    ]
    airports_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
    airports_df = pd.read_csv(airports_url, header=None, names=openflights_cols)

    # Filter to Alaska (approximate bounding box)
    ak_airports_df = airports_df[
        (airports_df['country'] == 'United States') &
        (airports_df['latitude'] >= 51) & (airports_df['latitude'] <= 72) &
        (airports_df['longitude'] >= -180) & (airports_df['longitude'] <= -129)
    ].copy()

    airports_gdf = gpd.GeoDataFrame(
        ak_airports_df,
        geometry=gpd.points_from_xy(ak_airports_df.longitude, ak_airports_df.latitude),
        crs='EPSG:4326',
    )
    print(f"OpenFlights airports in Alaska: {len(airports_gdf)}")

# Reproject to EPSG:3413
airports_3413 = airports_gdf.to_crs('EPSG:3413')

# Save
output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
airports_path = os.path.join(output_dir, 'airports_alaska.geojson')
airports_3413.to_file(airports_path, driver='GeoJSON')
print(f"Airports saved: {airports_path} ({len(airports_3413)} airports)")

## 9. Ports — AK DOT&PF

In [ ]:
# AK DOT&PF Port Locations
# Source: Alaska DOT&PF port/harbor data
#
# Port data can be obtained from:
# 1. AK DOT&PF GIS data portal
# 2. USACE Navigation Data Center
# 3. Manual compilation from known Alaska ports
#
# For now, we create a reference dataset of known Alaska ports with type
# classification (full port vs beach landing)

# Known major Alaska ports and beach landings
alaska_ports = [
    # Full ports (deep water, dock facilities)
    {'name': 'Port of Anchorage', 'lon': -149.893, 'lat': 61.238, 'type': 'port'},
    {'name': 'Port of Valdez', 'lon': -146.348, 'lat': 61.127, 'type': 'port'},
    {'name': 'Port of Seward', 'lon': -149.427, 'lat': 60.104, 'type': 'port'},
    {'name': 'Port of Homer', 'lon': -151.410, 'lat': 59.602, 'type': 'port'},
    {'name': 'Port of Kodiak', 'lon': -152.407, 'lat': 57.787, 'type': 'port'},
    {'name': 'Port of Dutch Harbor / Unalaska', 'lon': -166.542, 'lat': 53.879, 'type': 'port'},
    {'name': 'Port of Juneau', 'lon': -134.420, 'lat': 58.300, 'type': 'port'},
    {'name': 'Port of Ketchikan', 'lon': -131.646, 'lat': 55.342, 'type': 'port'},
    {'name': 'Port of Sitka', 'lon': -135.330, 'lat': 57.053, 'type': 'port'},
    {'name': 'Port of Skagway', 'lon': -135.314, 'lat': 59.451, 'type': 'port'},
    {'name': 'Port of Whittier', 'lon': -148.683, 'lat': 60.773, 'type': 'port'},
    {'name': 'Port of Dillingham', 'lon': -158.458, 'lat': 59.040, 'type': 'port'},
    {'name': 'Port of Bethel', 'lon': -161.756, 'lat': 60.793, 'type': 'port'},
    {'name': 'Port of Nome', 'lon': -165.406, 'lat': 64.501, 'type': 'port'},
    {'name': 'Port of Kotzebue', 'lon': -162.596, 'lat': 66.898, 'type': 'port'},
    {'name': 'Port of Naknek', 'lon': -156.997, 'lat': 58.727, 'type': 'port'},
    {'name': 'Port of Wrangell', 'lon': -132.376, 'lat': 56.471, 'type': 'port'},
    {'name': 'Port of Petersburg', 'lon': -132.955, 'lat': 56.812, 'type': 'port'},
    {'name': 'Port of Haines', 'lon': -135.445, 'lat': 59.236, 'type': 'port'},
    {'name': 'Port of Cordova', 'lon': -145.753, 'lat': 60.543, 'type': 'port'},
    # Beach landings (lighter facilities, seasonal)
    {'name': 'Barrow / Utqiagvik', 'lon': -156.789, 'lat': 71.291, 'type': 'beach_landing'},
    {'name': 'Point Hope', 'lon': -166.764, 'lat': 68.348, 'type': 'beach_landing'},
    {'name': 'Wainwright', 'lon': -159.953, 'lat': 70.638, 'type': 'beach_landing'},
    {'name': 'Kivalina', 'lon': -164.533, 'lat': 67.726, 'type': 'beach_landing'},
    {'name': 'Shishmaref', 'lon': -166.072, 'lat': 66.256, 'type': 'beach_landing'},
    {'name': 'Emmonak', 'lon': -164.523, 'lat': 62.777, 'type': 'beach_landing'},
    {'name': 'St. Michael', 'lon': -162.110, 'lat': 63.472, 'type': 'beach_landing'},
    {'name': 'Hooper Bay', 'lon': -166.097, 'lat': 61.531, 'type': 'beach_landing'},
    {'name': 'Kipnuk', 'lon': -164.031, 'lat': 59.933, 'type': 'beach_landing'},
    {'name': 'Togiak', 'lon': -160.377, 'lat': 59.063, 'type': 'beach_landing'},
]

ports_df = pd.DataFrame(alaska_ports)
ports_gdf = gpd.GeoDataFrame(
    ports_df,
    geometry=gpd.points_from_xy(ports_df.lon, ports_df.lat),
    crs='EPSG:4326',
)

# Reproject to EPSG:3413
ports_3413 = ports_gdf.to_crs('EPSG:3413')

# Save
output_dir = os.getenv('VECTOR_DIR', './vectors')
ports_path = os.path.join(output_dir, 'ports_alaska.geojson')
ports_3413.to_file(ports_path, driver='GeoJSON')
print(f"Ports saved: {ports_path}")
print(f"  Full ports: {len(ports_df[ports_df['type'] == 'port'])}")
print(f"  Beach landings: {len(ports_df[ports_df['type'] == 'beach_landing'])}")

## 10. Facilities — Bulk Fuel Sites

In [ ]:
# Bulk Fuel Facility Sites — reproject from CSV to EPSG:3413

# Path to the bulk fuel CSV (adjust if running from notebooks/ subdirectory)
csv_path = '../Utilities_Bulk_Fuel_Inventory.csv'
if not os.path.exists(csv_path):
    csv_path = 'Utilities_Bulk_Fuel_Inventory.csv'

bulk_fuel = pd.read_csv(csv_path, usecols=[
    'ASTFacilityID', 'ASTFacilityLongitude', 'ASTFacilityLatitude',
    'CommunityName', 'Delivery_method'
])

# Drop rows with missing coordinates
bulk_fuel = bulk_fuel.dropna(subset=['ASTFacilityLongitude', 'ASTFacilityLatitude'])

facilities_gdf = gpd.GeoDataFrame(
    bulk_fuel,
    geometry=gpd.points_from_xy(
        bulk_fuel['ASTFacilityLongitude'],
        bulk_fuel['ASTFacilityLatitude'],
    ),
    crs='EPSG:4326',
)

# Reproject to EPSG:3413
facilities_3413 = facilities_gdf.to_crs('EPSG:3413')

# Save
output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
facilities_path = os.path.join(output_dir, 'facilities_alaska.geojson')
facilities_3413.to_file(facilities_path, driver='GeoJSON')
print(f"Facilities saved: {facilities_path} ({len(facilities_3413)} sites)")

## 11. Monitor GEE Export Tasks & Alignment Verification

In [ ]:
# Monitor export tasks
import time

def monitor_tasks(tasks, poll_interval=30):
    """Monitor GEE export tasks until all complete."""
    print(f"Monitoring {len(tasks)} export tasks...")
    while True:
        statuses = {}
        for t in tasks:
            status = t.status()
            state = status['state']
            statuses[status['description']] = state

        completed = sum(1 for s in statuses.values() if s == 'COMPLETED')
        failed = sum(1 for s in statuses.values() if s == 'FAILED')
        running = sum(1 for s in statuses.values() if s in ('RUNNING', 'READY'))

        print(f"\r  Completed: {completed} | Running: {running} | Failed: {failed}", end='')

        if running == 0:
            print()
            break
        time.sleep(poll_interval)

    # Print final status
    print("\nFinal status:")
    for desc, state in statuses.items():
        symbol = '  ' if state == 'COMPLETED' else '  '
        print(f"  {symbol} {desc}: {state}")

    if failed > 0:
        print(f"\nWARNING: {failed} task(s) failed. Check GEE Tasks tab for details.")

# Uncomment to monitor (this will block until all exports complete):
# monitor_tasks(export_tasks)

In [ ]:
# Alignment Verification
# Run this AFTER downloading exported rasters from Google Drive to the rasters/ directory

import rasterio

raster_dir = os.getenv('RASTER_DIR', './rasters')
raster_files = {
    'LULC': os.path.join(raster_dir, 'lulc_alaska_modal.tif'),
    'Slope': os.path.join(raster_dir, 'slope_alaska.tif'),
    'DEM': os.path.join(raster_dir, 'dem_alaska.tif'),
    'Permafrost': os.path.join(raster_dir, 'permafrost_alaska.tif'),
    'Roads Presence': os.path.join(raster_dir, 'roads_presence_alaska.tif'),
    'Rivers': os.path.join(raster_dir, 'rivers_alaska.tif'),
}

print(f"{'Layer':<20} {'Shape':<20} {'CRS':<15} {'Res (m)':<12} {'Dtype':<10} {'Min':<10} {'Max':<10}")
print("-" * 97)

reference_crs = None
reference_transform = None
reference_shape = None
all_aligned = True

for name, path in raster_files.items():
    if not os.path.exists(path):
        print(f"{name:<20} FILE NOT FOUND: {path}")
        all_aligned = False
        continue

    with rasterio.open(path) as src:
        data = src.read(1)
        crs = str(src.crs)
        res = src.res
        shape = (src.height, src.width)
        dtype = str(src.dtypes[0])
        transform = src.transform

        # Check alignment
        if reference_crs is None:
            reference_crs = crs
            reference_transform = transform
            reference_shape = shape
        else:
            if crs != reference_crs:
                all_aligned = False
            if shape != reference_shape:
                all_aligned = False
            if transform != reference_transform:
                all_aligned = False

        valid = data[data != src.nodata] if src.nodata else data
        vmin = f"{np.nanmin(valid):.1f}" if len(valid) > 0 else "N/A"
        vmax = f"{np.nanmax(valid):.1f}" if len(valid) > 0 else "N/A"

        print(f"{name:<20} {str(shape):<20} {crs:<15} {res[0]:<12.1f} {dtype:<10} {vmin:<10} {vmax:<10}")

print()
if all_aligned:
    print("ALL RASTERS ALIGNED - CRS, extent, resolution, and pixel grid match.")
else:
    print("WARNING: Raster alignment issues detected. Check CRS, extent, and resolution.")